In [1]:
# Install dependencies
%pip install anthropic python-dotenv
%pip install dotenv
%pip install IPython


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# Create an API Client
from anthropic import Anthropic

client=Anthropic()
model="claude-haiku-4-5"


In [ ]:
# make a request
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]   
)

message.content[0].text 

In [ ]:
# Define helper functions to support multi-message converstation
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages
    )
    return message.content[0].text

In [ ]:
# Make a starting list of messages
messages = []

# Add the initial user message
add_user_message(messages, "Define quantum computing in one sentence.")

# Pass the list of messates into 'chat' to get an answer
answer = chat(messages)

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)

# Add in user's follow-up question
add_user_message(messages, "Write another sentence.")

# Pass the list of messates into 'chat' to get an answer
answer = chat(messages)

# Add in the answer to the follow up question
add_assistant_message(messages, answer)

messages


In [ ]:
# Chat Exercise
messages = []

while True:
    # Get user input
    user_input = input(">")
    print(">", user_input)

    # add user_messages to messages list
    add_user_message(messages, user_input)

    # call chat to get answer
    answer = chat(messages)

    # add answer to messages list
    add_assistant_message(messages, answer)

    # print answer
    print("---")
    print(answer)
    print("---")



In [ ]:
# Math Tutor Exercise (chat with System Prompts)
system_prompt = "You are a patient math tutor. Do not answer the question directly, but guilde the them to a solution step by step."

def chat_v2(messages, prompt=None):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages,
    }

    if prompt:
        params["system"] = prompt

    message = client.messages.create(**params)
    return message.content[0].text    
    
messages = []
add_user_message(messages,"How do I create an integral expression from a limit sum expression?")
answer = chat_v2(messages, system_prompt) # test with and without providing the system prompt
add_assistant_message(messages,answer)
print(answer)


In [ ]:
# system prompt exercise 2 - concise code example

messages = []

add_user_message(messages, "Write a Python function that checks a string for duplicate characters")
answer = chat_v2(messages, prompt="You are a Python engineer who rites very concise code")
add_assistant_message(messages, answer)
print(answer)

In [ ]:
# Temperature exercise

def chat_v3(messages, prompt=None, temperature=1.0):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages,
        "temperature": temperature
    }

    if prompt:
        params["system"] = prompt

    message = client.messages.create(**params)
    return message.content[0].text    

messages = []

add_user_message(messages, "Generate a one sentence movie idea")
answer = chat_v3(messages, temperature=1.0) # test with different temperatures between 0.0 and 1.0
add_assistant_message(messages, answer)
print(answer)

In [ ]:
# Raw Steaming Respoonse Exercise

message = []

add_user_message(messages, "Given me an example of a hidden variable theory related to the unification of quantum and relativistic physics")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True,
)

for event in stream:
    print(event)




In [ ]:
# Managed Steaming Respoonse Exercise

message = []

add_user_message(messages, "Given me an example of a hidden variable theory related to the unification of quantum and relativistic physics")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="") # outputs chuncks of reponse as they are received

stream.get_final_message() # collects entire final collection of messages so it can stored or consumed seperately

In [ ]:
#structured data example 1 (prefil and stop sequences)

messages = []

def chat_v4(messages, prompt=None, temperature=1.0, stop_sequences=None):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages,
        "temperature": temperature
    }

    if prompt:
        params["system"] = prompt

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text    

add_user_message(messages,"Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat_v4(messages, stop_sequences=["```"])
text

In [ ]:
# work with the JSON response from previous request

import json

json.loads(text.strip())

In [ ]:
#structured data example 2 (prefil and stop sequences)

messages = []

prompt = """
Generate thee different sample ASW CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all 3 commands in a single block without any comments:\n```bash")

text = chat_v4(messages, stop_sequences=["```"])
text.strip()

In [ ]:
from IPython.display import Markdown

Markdown(text)